In [1]:
%pip install -q chromadb ollama

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

data_folder = Path("data")
output_folder = data_folder / Path("artifacts")
Path.cwd()

WindowsPath('c:/stock_agent01/rag_exam')

In [3]:
import json

chunks = []
with(output_folder / "chunks.jsonl").open(encoding="utf-8") as file:
    for line in file:
        chunks.append(json.loads(line))

In [4]:
chunks_path = output_folder / "chunks.jsonl"

for chunk in chunks:
    if "selection" in chunk:
        chunk["section"] = chunk.pop("selection")

with chunks_path.open("w", encoding="utf-8") as file:
    for chunk in chunks:
        file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

In [5]:
import ollama

model = "qwen3-embedding:0.6b"

In [6]:
response = ollama.embed(model=model, input=chunks[0]["text"], truncate=False)
sample_vector = response.embeddings[0]

In [7]:
len(sample_vector)

1024

In [8]:
import chromadb

db_folder = Path("chroma_db").resolve()

client = chromadb.PersistentClient(path=str(db_folder))

In [9]:
collection_name = "skhynix_basic"
collection = client.get_or_create_collection(
    name=collection_name,
    embedding_function=None,
    configuration={"hnsw":{"space":"cosine"}}
)

In [10]:
old_ids = collection.get(include=[])["ids"]
if old_ids:
    collection.delete(ids=old_ids)

In [11]:
for start in range(0, len(chunks), 16):
    batch = chunks[start:start +16]
    ids=[]
    texts=[]
    metadata=[]
    for chunk in batch:
        ids.append(chunk["id"])
        texts.append(chunk["text"])
        metadata.append({
            "source": chunk["source"], "period": chunk["period"],
            "section": chunk["section"]
        })
    response = ollama.embed(model=model,input=texts,truncate=False)
    collection.upsert(
        ids=ids, documents=texts, metadatas=metadata,
        embeddings=response.embeddings
    )

In [12]:
{"입력 청크": len(chunks), "DB 청크": collection.count()}

{'입력 청크': 796, 'DB 청크': 796}

In [13]:
query_prefix="Instruct: Given a question, retrieve passages from Korean financial disclosures that answer the question.\nQuery:"

In [14]:
settings = {
    "model": model, "collection": collection_name,
    "query_prefix": query_prefix, "chunk_count": len(chunks)
}
with (output_folder / "index.json").open("w", encoding="utf-8") as file:
    json.dump(settings, file, ensure_ascii=False, indent=2)